In [1]:
%pip install -q pillow

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont

output_dir = Path("module-5/unit-2")
output_dir.mkdir(parents=True, exist_ok=True)

image_path = output_dir / "l2_1_sample_receipt.png"

image = Image.new("RGB", (900, 620), "white")
draw = ImageDraw.Draw(image)
font = ImageFont.load_default()

lines = [
    "SAMPLE RECEIPT - PRACTICE DATA ONLY",
    "Supplier: Example Office Supplies",
    "Date: 2026-06-25",
    "Item: Notebook Pack      12.99",
    "Item: Printer Paper      8.50",
    "Total:                   21.49",
    "",
    "Handwritten note: urgent reimbursement",
]

y = 60
for line in lines:
    draw.text((70, y), line, fill="black", font=font)
    y += 48

draw.rectangle((55, 45, 845, 540), outline="black", width=3)
draw.text((590, 500), "APPROVED?", fill="black", font=font)

image.save(image_path)

print(f"Sample image saved to: {image_path}")
print(f"Image size: {image.size}")

Sample image saved to: module-5/unit-2/l2_1_sample_receipt.png
Image size: (900, 620)


In [8]:
from pathlib import Path

from PIL import Image

image_path = Path("module-5/unit-2/l2_1_sample_receipt.png")

with Image.open(image_path) as img:
    width, height = img.size
    mode = img.mode
    image_format = img.format

file_size_kb = image_path.stat().st_size / 1024
megapixels = (width * height) / 1_000_000

preflight = {
    "file_name": image_path.name,
    "file_size_kb": round(file_size_kb, 2),
    "width": width,
    "height": height,
    "megapixels": round(megapixels, 2),
    "mode": mode,
    "format": image_format,
    "quality_check": "pass" if width >= 600 and height >= 400 else "review",
}

preflight

{'file_name': 'l2_1_sample_receipt.png',
 'file_size_kb': 12.81,
 'width': 900,
 'height': 620,
 'megapixels': 0.56,
 'mode': 'RGB',
 'format': 'PNG',
 'quality_check': 'pass'}

In [9]:
import json
from pathlib import Path

case = {
    "case_name": "sample_receipt",
    "input_type": "scanned_document",
    "visual_layout_matters": True,
    "text_can_be_extracted_first": False,
    "contains_sensitive_data": False,
    "image_quality": preflight["quality_check"],
    "reason": "The task may depend on receipt layout and handwritten note interpretation.",
}


def recommend_route(case):
    if case["contains_sensitive_data"]:
        return {
            "route": "human_review_or_approved_route",
            "reason": "Sensitive data needs an approved handling route before model processing.",
        }

    if case["image_quality"] != "pass":
        return {
            "route": "human_review_or_rescan",
            "reason": "Image quality is not good enough for reliable automated processing.",
        }

    if not case["visual_layout_matters"]:
        return {
            "route": "text_only",
            "reason": "The task does not require visual information.",
        }

    if case["text_can_be_extracted_first"]:
        return {
            "route": "ocr_first",
            "reason": "Text can be extracted before the LLM is used.",
        }

    return {
        "route": "multimodal_candidate",
        "reason": "Visual layout or image content is needed for the task.",
    }


decision = {
    "lesson": "L2.1",
    "preflight": preflight,
    "case": case,
    "recommendation": recommend_route(case),
}

decision_path = Path("module-5/unit-2/l2_1_modality_decision.json")
decision_path.write_text(json.dumps(decision, indent=2), encoding="utf-8")

print(f"Decision saved to: {decision_path}")
print(json.dumps(decision, indent=2))

Decision saved to: module-5/unit-2/l2_1_modality_decision.json
{
  "lesson": "L2.1",
  "preflight": {
    "file_name": "l2_1_sample_receipt.png",
    "file_size_kb": 12.81,
    "width": 900,
    "height": 620,
    "megapixels": 0.56,
    "mode": "RGB",
    "format": "PNG",
    "quality_check": "pass"
  },
  "case": {
    "case_name": "sample_receipt",
    "input_type": "scanned_document",
    "visual_layout_matters": true,
    "text_can_be_extracted_first": false,
    "contains_sensitive_data": false,
    "image_quality": "pass",
    "reason": "The task may depend on receipt layout and handwritten note interpretation."
  },
  "recommendation": {
    "route": "multimodal_candidate",
    "reason": "Visual layout or image content is needed for the task."
  }
}


In [10]:
import json
from pathlib import Path

case = {
    "case_name": "sample_receipt",
    "input_type": "scanned_document",
    "visual_layout_matters": True,
    "text_can_be_extracted_first": False,
    "contains_sensitive_data": False,
    "image_quality": preflight["quality_check"],
    "reason": "The task may depend on receipt layout and handwritten note interpretation.",
}


def recommend_route(case):
    if case["contains_sensitive_data"]:
        return {
            "route": "human_review_or_approved_route",
            "reason": "Sensitive data needs an approved handling route before model processing.",
        }

    if case["image_quality"] != "pass":
        return {
            "route": "human_review_or_rescan",
            "reason": "Image quality is not good enough for reliable automated processing.",
        }

    if not case["visual_layout_matters"]:
        return {
            "route": "text_only",
            "reason": "The task does not require visual information.",
        }

    if case["text_can_be_extracted_first"]:
        return {
            "route": "ocr_first",
            "reason": "Text can be extracted before the LLM is used.",
        }

    return {
        "route": "multimodal_candidate",
        "reason": "Visual layout or image content is needed for the task.",
    }


decision = {
    "lesson": "L2.1",
    "preflight": preflight,
    "case": case,
    "recommendation": recommend_route(case),
}

decision_path = Path("module-5/unit-2/l2_1_modality_decision.json")
decision_path.write_text(json.dumps(decision, indent=2), encoding="utf-8")

print(f"Decision saved to: {decision_path}")
print(json.dumps(decision, indent=2))

Decision saved to: module-5/unit-2/l2_1_modality_decision.json
{
  "lesson": "L2.1",
  "preflight": {
    "file_name": "l2_1_sample_receipt.png",
    "file_size_kb": 12.81,
    "width": 900,
    "height": 620,
    "megapixels": 0.56,
    "mode": "RGB",
    "format": "PNG",
    "quality_check": "pass"
  },
  "case": {
    "case_name": "sample_receipt",
    "input_type": "scanned_document",
    "visual_layout_matters": true,
    "text_can_be_extracted_first": false,
    "contains_sensitive_data": false,
    "image_quality": "pass",
    "reason": "The task may depend on receipt layout and handwritten note interpretation."
  },
  "recommendation": {
    "route": "multimodal_candidate",
    "reason": "Visual layout or image content is needed for the task."
  }
}
